# HDFS FILESYSTEM SHELL LAB — GOOGLE COLAB
**Môn:** Big Data & Cloud Computing  
**Chủ đề:** Hadoop FileSystem Shell / HDFS Shell  
**Môi trường:** Google Colab + Apache Hadoop 3.5.0

> Bài thực hành này tập trung riêng vào các lệnh FileSystem Shell của Hadoop.  
> Sinh viên không cần cài Hadoop trên máy cá nhân.

## Mục tiêu
Sau bài thực hành, sinh viên có thể:
1. Phân biệt **Local File System** và **HDFS**.
2. Sử dụng `hdfs dfs` / `hadoop fs`.
3. Tạo và quản lý thư mục HDFS.
4. Upload, đọc, append, copy, move và download file.
5. Kiểm tra dung lượng, số lượng file, metadata và checksum.
6. Tìm kiếm file trên HDFS.
7. Thực hành quyền truy cập cơ bản.
8. Xóa file/thư mục đúng cách.
9. Hoàn thành một workflow quản lý dữ liệu HDFS từ đầu đến cuối.

## Tài liệu chuẩn
Apache Hadoop FileSystem Shell:
`https://hadoop.apache.org/docs/current/hadoop-project-dist/hadoop-common/FileSystemShell.html`

# CHEAT SHEET — CÁC LỆNH SẼ THỰC HÀNH

| Nhóm | Lệnh |
|---|---|
| Trợ giúp | `-help`, `-usage` |
| Thư mục | `-mkdir`, `-ls`, `-find` |
| Local → HDFS | `-put`, `-copyFromLocal`, `-moveFromLocal` |
| Đọc file | `-cat`, `-head`, `-tail`, `-text` |
| Chỉnh sửa | `-appendToFile`, `-touch`, `-touchz` |
| Sao chép / đổi tên | `-cp`, `-mv` |
| Thống kê | `-du`, `-df`, `-count`, `-stat`, `-checksum` |
| HDFS → Local | `-get`, `-copyToLocal`, `-getmerge` |
| Quyền | `-chmod`, `-chgrp` |
| Replication | `-setrep` |
| Kiểm tra | `-test` |
| Xóa | `-rm`, `-rmdir` |

# PHẦN 0 — KHỞI TẠO HADOOP

Cell này:
- cài SSH,
- tải Hadoop 3.5.0,
- cấu hình pseudo-distributed HDFS,
- format NameNode lần đầu,
- khởi động HDFS.

> Nếu Colab Runtime bị reset, chạy lại từ Phần 0.

In [ ]:
%%bash
set -e

HADOOP_VERSION=3.5.0
HADOOP_HOME=/content/hadoop-${HADOOP_VERSION}

echo "=== Java ==="
java -version

echo "=== Install SSH ==="
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openssh-server rsync > /dev/null

echo "=== Download Hadoop ==="
if [ ! -d "$HADOOP_HOME" ]; then
  FILE=hadoop-${HADOOP_VERSION}.tar.gz
  URL1=https://downloads.apache.org/hadoop/common/hadoop-${HADOOP_VERSION}/${FILE}
  URL2=https://archive.apache.org/dist/hadoop/common/hadoop-${HADOOP_VERSION}/${FILE}
  wget -q "$URL1" -O /content/$FILE || wget -q "$URL2" -O /content/$FILE
  tar -xzf /content/$FILE -C /content
fi

JAVA_HOME=$(dirname "$(dirname "$(readlink -f "$(command -v java)")")")
CONF=$HADOOP_HOME/etc/hadoop

echo "=== Passwordless SSH ==="
mkdir -p ~/.ssh
chmod 700 ~/.ssh
if [ ! -f ~/.ssh/id_rsa ]; then
  ssh-keygen -q -t rsa -N "" -f ~/.ssh/id_rsa
fi
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
sort -u ~/.ssh/authorized_keys -o ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys
service ssh restart > /dev/null
ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null || true

cat > "$CONF/core-site.xml" <<'EOF'
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
EOF

cat > "$CONF/hdfs-site.xml" <<'EOF'
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:/content/hadoop_data/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:/content/hadoop_data/datanode</value>
  </property>
</configuration>
EOF

if ! grep -q "^export JAVA_HOME=" "$CONF/hadoop-env.sh"; then
  echo "export JAVA_HOME=$JAVA_HOME" >> "$CONF/hadoop-env.sh"
fi

$HADOOP_HOME/sbin/stop-dfs.sh >/dev/null 2>&1 || true

mkdir -p /content/hadoop_data
if [ ! -d /content/hadoop_data/namenode/current ]; then
  $HADOOP_HOME/bin/hdfs namenode -format -force -nonInteractive > /dev/null
fi

$HADOOP_HOME/sbin/start-dfs.sh

echo "=== Hadoop processes ==="
jps


In [ ]:
import os
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["HADOOP_CONF_DIR"] = "/content/hadoop-3.5.0/etc/hadoop"
os.environ["PATH"] = (
    "/content/hadoop-3.5.0/bin:"
    "/content/hadoop-3.5.0/sbin:" +
    os.environ["PATH"]
)

!hadoop version | head -n 5
!jps

## Kiểm tra kỳ vọng
`jps` nên có:
```text
NameNode
DataNode
SecondaryNameNode
```

### Câu hỏi 0
- NameNode quản lý gì?
- DataNode lưu gì?
- Vì sao notebook này chỉ dùng replication = 1?

# BÀI 1 — HDFS SHELL CƠ BẢN

Apache hỗ trợ:
```bash
hadoop fs <args>
```

Khi dùng HDFS, có thể dùng:
```bash
hdfs dfs <args>
```

Trong notebook này thống nhất dùng `hdfs dfs`.

## 1.1 Xem help

In [ ]:
!hdfs dfs -help | head -n 40

## 1.2 Help cho một lệnh cụ thể

In [ ]:
!hdfs dfs -help ls

### Bài tập 1
Dùng `-help` để xem cú pháp của:
- `mkdir`
- `put`
- `get`
- `rm`

In [ ]:
# TODO - BÀI TẬP 1

# BÀI 2 — TẠO VÀ LIỆT KÊ THƯ MỤC

## 2.1 Tạo HDFS home directory

In [ ]:
!hdfs dfs -mkdir -p /user/student
!hdfs dfs -ls /user

## 2.2 Tạo cây thư mục

In [ ]:
!hdfs dfs -mkdir -p /user/student/lab/input
!hdfs dfs -mkdir -p /user/student/lab/output
!hdfs dfs -mkdir -p /user/student/lab/archive
!hdfs dfs -ls -R /user/student/lab

## 2.3 `ls -h` và `ls -R`

In [ ]:
!hdfs dfs -ls -h /user/student/lab
!hdfs dfs -ls -R /user/student/lab

### Bài tập 2
Tạo:
```text
/user/student/project/
├── raw/
├── processed/
└── backup/
```

Sau đó dùng `ls -R` để kiểm tra.

In [ ]:
# TODO - BÀI TẬP 2

# BÀI 3 — LOCAL FILE SYSTEM → HDFS

## 3.1 Tạo file local

In [ ]:
sample = '''Hadoop Distributed File System
HDFS stores files in blocks
NameNode manages metadata
DataNode stores blocks
Hadoop FileSystem Shell is useful
'''
open("/content/hdfs_sample.txt","w").write(sample)

!ls -lh /content/hdfs_sample.txt
!cat /content/hdfs_sample.txt

## 3.2 Upload bằng `put`

In [ ]:
!hdfs dfs -put -f /content/hdfs_sample.txt /user/student/lab/input/
!hdfs dfs -ls -h /user/student/lab/input

## 3.3 `copyFromLocal`

In [ ]:
open("/content/data2.txt","w").write("second local file\n")
!hdfs dfs -copyFromLocal -f /content/data2.txt /user/student/lab/input/
!hdfs dfs -ls /user/student/lab/input

### Bài tập 3
1. Tạo `student.txt` chứa MSSV, họ tên, lớp.
2. Upload bằng `put`.
3. Tạo `note.txt`.
4. Upload bằng `copyFromLocal`.
5. Dùng `ls -h` kiểm tra.

In [ ]:
# TODO - BÀI TẬP 3

# BÀI 4 — ĐỌC NỘI DUNG FILE

## 4.1 `cat`

In [ ]:
!hdfs dfs -cat /user/student/lab/input/hdfs_sample.txt

## 4.2 `head`

In [ ]:
!hdfs dfs -head /user/student/lab/input/hdfs_sample.txt

## 4.3 `tail`

In [ ]:
!hdfs dfs -tail /user/student/lab/input/hdfs_sample.txt

### Bài tập 4
Dùng `cat`, `head`, `tail` cho file `student.txt`.  
Ghi nhận sự khác nhau về kết quả.

In [ ]:
# TODO - BÀI TẬP 4

# BÀI 5 — APPEND VÀ TOUCH

## 5.1 `appendToFile`

In [ ]:
open("/content/append.txt","w").write("New line appended from local filesystem\n")
!hdfs dfs -appendToFile /content/append.txt /user/student/lab/input/hdfs_sample.txt
!hdfs dfs -cat /user/student/lab/input/hdfs_sample.txt

## 5.2 `touchz` — tạo file rỗng

In [ ]:
!hdfs dfs -touchz /user/student/lab/input/empty.txt
!hdfs dfs -ls -h /user/student/lab/input/empty.txt

## 5.3 `touch`

In [ ]:
!hdfs dfs -touch /user/student/lab/input/touched.txt
!hdfs dfs -ls /user/student/lab/input/touched.txt

### Bài tập 5
1. Tạo file local `append_student.txt`.
2. Append vào `student.txt`.
3. Tạo `done.flag` bằng `touchz`.
4. Kiểm tra kích thước `done.flag`.

In [ ]:
# TODO - BÀI TẬP 5

# BÀI 6 — COPY VÀ MOVE TRONG HDFS

## 6.1 `cp`

In [ ]:
!hdfs dfs -cp -f   /user/student/lab/input/hdfs_sample.txt   /user/student/lab/archive/hdfs_sample_backup.txt

!hdfs dfs -ls /user/student/lab/archive

## 6.2 `mv`

In [ ]:
!hdfs dfs -cp -f /user/student/lab/input/data2.txt /user/student/lab/input/data2_copy.txt
!hdfs dfs -mv /user/student/lab/input/data2_copy.txt /user/student/lab/archive/data2_moved.txt
!hdfs dfs -ls -R /user/student/lab

### Bài tập 6
1. Copy `student.txt` vào `archive/`.
2. Đổi tên thành `student_backup.txt`.
3. Xác nhận file gốc vẫn tồn tại.

In [ ]:
# TODO - BÀI TẬP 6

# BÀI 7 — THỐNG KÊ DUNG LƯỢNG

## 7.1 `du`

In [ ]:
!hdfs dfs -du -h /user/student/lab/input

## 7.2 `du -s`

In [ ]:
!hdfs dfs -du -s -h /user/student/lab

## 7.3 `df`

In [ ]:
!hdfs dfs -df -h /

## 7.4 `count`

In [ ]:
!hdfs dfs -count -h -v /user/student/lab

### Câu hỏi 7
1. `du` cho biết dung lượng của path; hai cột `SIZE` và `DISK_SPACE_CONSUMED` khác nhau thế nào khi replication > 1?
2. `df` cho biết thông tin gì về toàn filesystem?
3. `count` trả về những thông tin nào?


# BÀI 8 — METADATA: STAT VÀ CHECKSUM

## 8.1 `stat`

In [ ]:
!hdfs dfs -stat "name=%n | size=%b | replication=%r | block=%o"   /user/student/lab/input/hdfs_sample.txt

## 8.2 `checksum`

In [ ]:
!hdfs dfs -checksum /user/student/lab/input/hdfs_sample.txt

### Bài tập 8
Dùng `stat` để lấy:
- tên file,
- kích thước,
- replication factor

cho `student.txt`.

In [ ]:
# TODO - BÀI TẬP 8

# BÀI 9 — TÌM KIẾM FILE

## 9.1 `find`

In [ ]:
!hdfs dfs -find /user/student/lab -name "*.txt" -print

## 9.2 Tìm theo tên

In [ ]:
!hdfs dfs -find /user/student/lab -name "hdfs*" -print

### Bài tập 9
1. Tìm tất cả file `.txt`.
2. Tìm file bắt đầu bằng `student`.
3. Tìm file `done.flag`.

In [ ]:
# TODO - BÀI TẬP 9

# BÀI 10 — KIỂM TRA ĐIỀU KIỆN BẰNG `test`

`test` hữu ích trong shell script.

Ví dụ:
- `-e`: path tồn tại
- `-d`: là directory
- `-f`: là file
- `-z`: file có size = 0

In [ ]:
%%bash
if hdfs dfs -test -e /user/student/lab/input/hdfs_sample.txt; then
  echo "File exists"
else
  echo "File does not exist"
fi

if hdfs dfs -test -z /user/student/lab/input/empty.txt; then
  echo "empty.txt is zero bytes"
fi

### Bài tập 10
Viết shell kiểm tra:
1. `/user/student/project/raw` có tồn tại không.
2. `done.flag` có size bằng 0 không.

In [ ]:
# TODO - BÀI TẬP 10

# BÀI 11 — HDFS → LOCAL FILE SYSTEM

## 11.1 `get`

In [ ]:
!rm -f /content/downloaded_hdfs_sample.txt
!hdfs dfs -get   /user/student/lab/input/hdfs_sample.txt   /content/downloaded_hdfs_sample.txt

!cat /content/downloaded_hdfs_sample.txt

## 11.2 `copyToLocal`

In [ ]:
!rm -f /content/data2_download.txt
!hdfs dfs -copyToLocal   /user/student/lab/input/data2.txt   /content/data2_download.txt

!cat /content/data2_download.txt

## 11.3 `getmerge`

In [ ]:
!hdfs dfs -mkdir -p /user/student/lab/merge
!hdfs dfs -cp -f /user/student/lab/input/hdfs_sample.txt /user/student/lab/merge/a.txt
!hdfs dfs -cp -f /user/student/lab/input/data2.txt /user/student/lab/merge/b.txt

!rm -f /content/merged.txt
!hdfs dfs -getmerge -nl /user/student/lab/merge /content/merged.txt
!cat /content/merged.txt

### Bài tập 11
1. Download `student.txt` về `/content/student_download.txt`.
2. Tạo 3 file HDFS nhỏ.
3. Dùng `getmerge -nl` ghép thành một file local.

In [ ]:
# TODO - BÀI TẬP 11

# BÀI 12 — QUYỀN TRUY CẬP CƠ BẢN

## 12.1 Xem permission bằng `ls`

In [ ]:
!hdfs dfs -ls /user/student/lab/input/hdfs_sample.txt

## 12.2 `chmod`

In [ ]:
!hdfs dfs -chmod 640 /user/student/lab/input/hdfs_sample.txt
!hdfs dfs -ls /user/student/lab/input/hdfs_sample.txt

### Câu hỏi 12
Giải thích `640`:
```text
owner = ?
group = ?
others = ?
```

# BÀI 13 — REPLICATION

`setrep` đặt replication factor mong muốn cho file. NameNode sẽ điều phối việc tăng/giảm số replica; lệnh không tự tạo thêm DataNode.

Google Colab chỉ có **một DataNode**, vì vậy bài chạy với replication = 1. Trên cụm này, đặt replication = 3 sẽ tạo trạng thái **under-replicated**, không tạo được ba bản sao vật lý.


In [ ]:
!hdfs dfs -setrep -w 1 /user/student/lab/input/hdfs_sample.txt
!hdfs dfs -stat "replication=%r" /user/student/lab/input/hdfs_sample.txt

### Câu hỏi 13
Nếu cluster có 3 DataNode và chạy:
```bash
hdfs dfs -setrep 3 /data/file.txt
```
thì ý nghĩa là gì?

# BÀI 14 — XÓA FILE VÀ THƯ MỤC

## 14.1 `rm`

In [ ]:
!hdfs dfs -touchz /user/student/lab/input/to_delete.txt
!hdfs dfs -rm /user/student/lab/input/to_delete.txt

## 14.2 `rm -r`

In [ ]:
!hdfs dfs -mkdir -p /user/student/lab/temp/a/b
!hdfs dfs -touchz /user/student/lab/temp/a/test.txt
!hdfs dfs -rm -r /user/student/lab/temp

## 14.3 `rmdir` — chỉ xóa thư mục rỗng

In [ ]:
!hdfs dfs -mkdir -p /user/student/lab/empty_dir
!hdfs dfs -rmdir /user/student/lab/empty_dir

### Bài tập 14
1. Tạo `trash_test/`.
2. Tạo 2 file trong đó.
3. Thử `rmdir trash_test` và quan sát lỗi.
4. Sau đó dùng `rm -r`.

In [ ]:
# TODO - BÀI TẬP 14

# BÀI 15 — BÀI TỔNG HỢP: HDFS FILE LIFECYCLE

## Tình huống
Anh/chị nhận một dataset local `transactions.csv`.

### Yêu cầu
1. Tạo `/user/student/final/raw`, `/processed`, `/backup`.
2. Upload `transactions.csv` vào `raw`.
3. Dùng `ls -h`.
4. Dùng `cat` kiểm tra dữ liệu.
5. Copy sang `backup`.
6. Dùng `stat` xem metadata.
7. Dùng `checksum`.
8. Dùng `du -h`.
9. Dùng `count -h -v`.
10. Tìm file bằng `find`.
11. Download file backup về local.
12. Tạo `SUCCESS.flag` bằng `touchz`.
13. Kiểm tra `SUCCESS.flag` bằng `test -z`.
14. Xóa thư mục `processed` nếu rỗng.

## Dữ liệu mẫu

In [ ]:
transactions = '''transaction_id,customer,city,amount
T001,C01,HCMC,1200
T002,C02,Hanoi,800
T003,C01,HCMC,600
T004,C03,Danang,1500
T005,C02,Hanoi,900
'''
open("/content/transactions.csv","w").write(transactions)
!cat /content/transactions.csv

In [ ]:
# TODO - BÀI TỔNG HỢP

# CÂU HỎI CỦNG CỐ

1. `hadoop fs` và `hdfs dfs` khác nhau thế nào?
2. Local path và HDFS path khác nhau ở đâu?
3. `put`/`copyFromLocal` và `get`/`copyToLocal` có quan hệ gì?
4. `cp` khác `get` thế nào?
5. `du`, `df` và `count` trả lời ba câu hỏi khác nhau nào?
6. `stat`, `ls` và `checksum` dùng cho những mục đích nào?
7. `find` và `test` hữu ích trong workflow tự động ra sao?
8. `rm` khác `rmdir` thế nào?
9. `setrep` thay đổi điều gì, và vì sao nó không tạo thêm DataNode?
10. Vì sao replication = 3 không thể đạt đủ trên single-node Colab?


# THU BÀI — RUBRIC 10 ĐIỂM

| Nội dung | Điểm |
|---|---:|
| Khởi tạo HDFS | 0.5 |
| mkdir / ls | 0.75 |
| put / copyFromLocal | 1.0 |
| cat / head / tail | 0.75 |
| append / touch | 0.5 |
| cp / mv | 0.75 |
| du / df / count | 1.0 |
| stat / checksum / find / test | 1.25 |
| get / getmerge | 1.0 |
| chmod / setrep / delete | 0.75 |
| Bài tổng hợp | 1.75 |

# GỢI Ý CHO SINH VIÊN KHÁ/GIỎI

Tìm hiểu thêm trong Apache FileSystem Shell:
- `getfacl`, `setfacl`
- `getfattr`, `setfattr`
- `truncate`
- `concat`
- snapshot commands
- trash / expunge

Không bắt buộc trong bài cơ bản.